# 02. 데이터 전처리 (Preprocessing)

**실행 환경:** 로컬 VSCode  
**목적:** 텍스트 정제, 레이블 통합, 학습 데이터셋 생성

---

## 전처리 파이프라인

```
Raw Data → 텍스트 정제 → 레이블 매핑 → 데이터 통합 → Train/Val/Test 분할 → 저장
```

In [ ]:
import os
import re
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 (Kaggle / Local 자동 감지) ──
if os.path.exists('/kaggle/input'):
    BASE_PATH = Path('/kaggle/input/civilcomplaint-raw')
    OUTPUT_PATH = Path('/kaggle/working/data/processed')
    IS_KAGGLE = True
else:
    BASE_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'raw'
    OUTPUT_PATH = Path.home() / 'CIVILCOMPLAINT' / 'data' / 'processed'
    IS_KAGGLE = False

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

CALLCENTER_PATH = BASE_PATH / 'callcenter_qa'
DIALOGUE_PATH = BASE_PATH / 'korean_dialogue'
LLM_PATH = BASE_PATH / 'llm_instruction'

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"경로 확인: {BASE_PATH.exists()}")

---
## 1. 텍스트 정제 함수

In [2]:
class TextPreprocessor:
    """한국어 민원 텍스트 전처리기"""
    
    def __init__(self):
        # 개인정보 패턴
        self.phone_pattern = re.compile(r'\d{2,3}[-.]?\d{3,4}[-.]?\d{4}')
        self.email_pattern = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
        self.ssn_pattern = re.compile(r'\d{6}[-]?\d{7}')  # 주민번호
        self.card_pattern = re.compile(r'\d{4}[-]?\d{4}[-]?\d{4}[-]?\d{4}')  # 카드번호
        
        # 특수문자 패턴
        self.html_pattern = re.compile(r'<[^>]+>')
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        self.emoji_pattern = re.compile(
            "["
            u"\U0001F600-\U0001F64F"  # 이모티콘
            u"\U0001F300-\U0001F5FF"  # 기호
            u"\U0001F680-\U0001F6FF"  # 교통
            u"\U0001F1E0-\U0001F1FF"  # 국기
            "]+", 
            flags=re.UNICODE
        )
        
        # 반복 문자 패턴 (ㅋㅋㅋㅋ, ㅎㅎㅎㅎ, ㅠㅠㅠㅠ 등)
        self.repeat_pattern = re.compile(r'(.)\1{2,}')
    
    def mask_personal_info(self, text):
        """개인정보 마스킹"""
        text = self.phone_pattern.sub('[전화번호]', text)
        text = self.email_pattern.sub('[이메일]', text)
        text = self.ssn_pattern.sub('[주민번호]', text)
        text = self.card_pattern.sub('[카드번호]', text)
        return text
    
    def clean_text(self, text):
        """텍스트 정제"""
        if pd.isna(text) or not isinstance(text, str):
            return ''
        
        text = text.strip()
        if not text:
            return ''
        
        # HTML 태그 제거
        text = self.html_pattern.sub('', text)
        
        # URL 제거
        text = self.url_pattern.sub('', text)
        
        # 이모지 제거
        text = self.emoji_pattern.sub('', text)
        
        # 개인정보 마스킹
        text = self.mask_personal_info(text)
        
        # 반복 문자 축소 (ㅋㅋㅋㅋ → ㅋㅋ)
        text = self.repeat_pattern.sub(r'\1\1', text)
        
        # 특수문자 정리 (한글, 영문, 숫자, 기본 문장부호만 유지)
        text = re.sub(r'[^가-힣a-zA-Z0-9\s.,!?\-()\[\]]', ' ', text)
        
        # 연속 공백 정리
        text = re.sub(r'\s+', ' ', text)
        
        return text.strip()
    
    def is_valid_text(self, text, min_len=5, max_len=500):
        """텍스트 유효성 검사"""
        if not text or not isinstance(text, str):
            return False
        length = len(text)
        return min_len <= length <= max_len

# 전처리기 인스턴스
preprocessor = TextPreprocessor()

# 테스트
test_cases = [
    "안녕하세요 010-1234-5678로 연락주세요 😀 <br>감사합니다",
    "ㅋㅋㅋㅋㅋ 진짜요????",
    "   공백만 있는 텍스트   ",
]
for tc in test_cases:
    print(f"원본: {tc!r}")
    print(f"정제: {preprocessor.clean_text(tc)!r}")
    print()

원본: '안녕하세요 010-1234-5678로 연락주세요 😀 <br>감사합니다'
정제: '안녕하세요 [전화번호]로 연락주세요 감사합니다'

원본: 'ㅋㅋㅋㅋㅋ 진짜요????'
정제: '진짜요??'

원본: '   공백만 있는 텍스트   '
정제: '공백만 있는 텍스트'



---
## 2. 데이터 로드 및 전처리

In [3]:
def load_callcenter_data(path):
    """콜센터 JSON 데이터 로드"""
    all_data = []
    json_files = list(path.rglob('*.json'))
    
    for file in tqdm(json_files, desc='Loading callcenter'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    all_data.extend(data)
                else:
                    all_data.append(data)
        except:
            pass
    
    return pd.DataFrame(all_data)

def load_dialogue_data(path):
    """대화 XLSX 데이터 로드"""
    all_dfs = []
    xlsx_files = list(path.rglob('*.xlsx'))
    
    for file in tqdm(xlsx_files, desc='Loading dialogue'):
        try:
            df = pd.read_excel(file)
            all_dfs.append(df)
        except:
            pass
    
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()

# 데이터 로드
print("데이터 로드 중...")
df_callcenter = load_callcenter_data(CALLCENTER_PATH)
df_dialogue = load_dialogue_data(DIALOGUE_PATH)

print(f"\nCallcenter: {len(df_callcenter):,} rows")
print(f"Dialogue: {len(df_dialogue):,} rows")

데이터 로드 중...



Loading callcenter:   0%|          | 0/50 [00:00<?, ?it/s]


Loading callcenter:   4%|▍         | 2/50 [00:00<00:05,  8.16it/s]


Loading callcenter:   6%|▌         | 3/50 [00:01<00:34,  1.37it/s]


Loading callcenter:  10%|█         | 5/50 [00:02<00:21,  2.08it/s]

...


Loading dialogue:  69%|██████▉   | 9/13 [00:07<00:03,  1.08it/s]


Loading dialogue:  77%|███████▋  | 10/13 [00:08<00:02,  1.08it/s]


Loading dialogue:  85%|████████▍ | 11/13 [00:10<00:02,  1.16s/it]


Loading dialogue: 100%|██████████| 13/13 [00:10<00:00,  1.22it/s]


Callcenter: 2,003,458 rows
Dialogue: 90,413 rows


---
## 3. 분류 모델용 데이터셋 생성

통합 스키마: `{text, domain, category, intent, source}`

In [4]:
def process_callcenter_for_classification(df):
    """콜센터 데이터 → 분류용 포맷 변환 (화자 기반 텍스트 추출 개선)"""
    
    # Q(질문)만 필터링
    df_q = df[df['QA'] == 'Q'].copy()
    print(f"Q 필터링: {len(df_q):,}건")
    
    # 텍스트 컬럼 정리 (strip 적용하여 공백만 있는 값도 빈값 처리)
    text_cols = ['고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변']
    for col in text_cols:
        if col in df_q.columns:
            df_q[col] = df_q[col].fillna('').astype(str).str.strip()
    
    # 화자 기반 벡터화 텍스트 추출
    is_customer = df_q['화자'] == '고객'
    
    # 고객 발화: 고객질문 → 고객답변 fallback
    customer_text = df_q['고객질문(요청)'].where(
        is_customer & (df_q['고객질문(요청)'].str.len() > 0),
        df_q['고객답변'].where(
            is_customer & (df_q['고객답변'].str.len() > 0),
            ''
        )
    )
    
    # 상담사 발화(QA='Q'): 상담사질문 → 상담사답변 fallback
    agent_text = df_q['상담사질문(요청)'].where(
        ~is_customer & (df_q['상담사질문(요청)'].str.len() > 0),
        df_q['상담사답변'].where(
            ~is_customer & (df_q['상담사답변'].str.len() > 0),
            ''
        )
    )
    
    df_q['text_raw'] = customer_text.where(is_customer, agent_text)
    
    # 텍스트 정제
    print("텍스트 정제 중...")
    df_q['text'] = df_q['text_raw'].apply(preprocessor.clean_text)
    
    # 유효한 텍스트만 필터링 (길이 5~500)
    df_q['text_len'] = df_q['text'].str.len()
    df_valid = df_q[(df_q['text_len'] >= 5) & (df_q['text_len'] <= 500)].copy()
    
    print(f"유효한 텍스트: {len(df_valid):,}건 ({len(df_valid)/len(df_q)*100:.1f}%)")
    print(f"  (min_len=5 미만 제외: {(df_q['text_len'] < 5).sum():,}건)")
    print(f"  (max_len=500 초과 제외: {(df_q['text_len'] > 500).sum():,}건)")
    
    # 화자별 추출 통계
    if '화자' in df_valid.columns:
        print(f"\n화자별 유효 텍스트:")
        print(df_valid['화자'].value_counts())
    
    # 필요한 컬럼만 선택 및 이름 변경
    result = df_valid[['text', '도메인', '카테고리', '고객의도']].copy()
    result.columns = ['text', 'domain', 'category', 'intent']
    result['source'] = 'callcenter'
    
    # 중복 제거
    before = len(result)
    result = result.drop_duplicates(subset=['text'])
    print(f"중복 제거: {before:,} → {len(result):,}건")
    
    return result

# 콜센터 데이터 처리
df_callcenter_processed = process_callcenter_for_classification(df_callcenter)
print(f"\n처리된 콜센터 데이터: {len(df_callcenter_processed):,} rows")

Q 필터링: 1,014,311건


텍스트 정제 중...


유효한 텍스트: 1,009,045건 (99.5%)
  (min_len=5 미만 제외: 5,266건)
  (max_len=500 초과 제외: 0건)

화자별 유효 텍스트:
화자
상담사    621979
고객     387066
Name: count, dtype: int64


중복 제거: 1,009,045 → 697,775건

처리된 콜센터 데이터: 697,775 rows


In [5]:
def process_dialogue_for_classification(df):
    """대화 데이터 → 분류용 포맷 변환 (벡터화 처리)"""
    
    text_col = 'SENTENCE' if 'SENTENCE' in df.columns else None
    domain_col = 'DOMAIN' if 'DOMAIN' in df.columns else None
    category_col = 'CATEGORY' if 'CATEGORY' in df.columns else None
    intent_col = 'MAIN' if 'MAIN' in df.columns else None
    qa_col = 'QA' if 'QA' in df.columns else None
    speaker_col = 'SPEAKER' if 'SPEAKER' in df.columns else None
    
    if not text_col:
        print("텍스트 컬럼을 찾을 수 없습니다.")
        return pd.DataFrame()
    
    # Q(질문)만 필터링
    if qa_col:
        df = df[df[qa_col] == 'Q'].copy()
    elif speaker_col:
        df = df[df[speaker_col] == '고객'].copy()
    
    print(f"Q 필터링 후: {len(df):,}건")
    
    # 벡터화 텍스트 정제
    df['text'] = df[text_col].fillna('').apply(preprocessor.clean_text)
    df['text_len'] = df['text'].str.len()
    
    # 유효성 필터 (5~500자)
    df_valid = df[(df['text_len'] >= 5) & (df['text_len'] <= 500)].copy()
    print(f"유효한 텍스트: {len(df_valid):,}건 ({len(df_valid)/len(df)*100:.1f}%)")
    
    # 결과 구성
    result = pd.DataFrame({
        'text': df_valid['text'].values,
        'domain': df_valid[domain_col].fillna('unknown').values if domain_col else 'unknown',
        'category': df_valid[category_col].fillna('unknown').values if category_col else 'unknown',
        'intent': df_valid[intent_col].fillna('unknown').values if intent_col else 'unknown',
        'source': 'dialogue'
    })
    
    # 중복 제거
    before = len(result)
    result = result.drop_duplicates(subset=['text'])
    print(f"중복 제거: {before:,} → {len(result):,}건")
    
    return result

# 대화 데이터 처리
df_dialogue_processed = process_dialogue_for_classification(df_dialogue)
print(f"처리된 대화 데이터: {len(df_dialogue_processed):,} rows")

Q 필터링 후: 51,065건


유효한 텍스트: 50,970건 (99.8%)
중복 제거: 50,970 → 47,039건
처리된 대화 데이터: 47,039 rows


In [6]:
# 데이터 통합
df_classification = pd.concat([df_callcenter_processed, df_dialogue_processed], ignore_index=True)

# 중복 제거
before_dedup = len(df_classification)
df_classification = df_classification.drop_duplicates(subset=['text'])
after_dedup = len(df_classification)

print(f"통합 데이터: {before_dedup:,} → 중복 제거 후: {after_dedup:,}")
print(f"\n도메인 분포:")
print(df_classification['domain'].value_counts())

통합 데이터: 744,814 → 중복 제거 후: 744,372

도메인 분포:
domain
K쇼핑       371210
질병관리본부    172994
금융/보험     101660
다산콜센터      51911
의복의류점       7776
음식점         7660
소매          7616
생활서비스       6455
부동산업        4494
카페          3919
숙박          3417
학원          2733
관광여가오락      2241
부동산          286
Name: count, dtype: int64


### 3-1. Intent 정제 및 클래스 불균형 분석

- `intent`가 35K+ 클래스로 대부분 고유값이거나 빈값 → 상위 빈출 intent만 유지, 나머지는 `기타`로 통합
- `domain` 클래스 간 최대 300배 이상 불균형 → class_weight 산출

In [7]:
# Intent 정제: 빈값/희소값 → '기타'로 통합
INTENT_MIN_COUNT = 50  # 최소 50건 이상인 intent만 유지

intent_counts = df_classification['intent'].value_counts()
valid_intents = set(intent_counts[intent_counts >= INTENT_MIN_COUNT].index) - {'', 'unknown'}

print(f"=== Intent 정제 ===")
print(f"  원본 intent 종류: {df_classification['intent'].nunique():,}개")
print(f"  {INTENT_MIN_COUNT}건 이상 intent: {len(valid_intents)}개")

# 빈값 또는 희소 intent를 '기타'로 변환
df_classification['intent_clean'] = df_classification['intent'].apply(
    lambda x: x if x in valid_intents else '기타'
)

print(f"  정제 후 intent 종류: {df_classification['intent_clean'].nunique()}개")
print(f"\n상위 intent 분포:")
print(df_classification['intent_clean'].value_counts().head(15))

# 클래스 불균형 분석 (domain 기준)
print(f"\n=== Domain 클래스 불균형 분석 ===")
domain_counts = df_classification['domain'].value_counts()
max_count = domain_counts.max()
min_count = domain_counts.min()
print(f"  최대: {domain_counts.idxmax()} ({max_count:,}건)")
print(f"  최소: {domain_counts.idxmin()} ({min_count:,}건)")
print(f"  불균형 비율: {max_count / min_count:.0f}:1")

# sklearn class_weight 산출 (학습 시 사용)
from sklearn.utils.class_weight import compute_class_weight

domain_classes = np.sort(df_classification['domain'].unique())
class_weights_array = compute_class_weight(
    'balanced',
    classes=domain_classes,
    y=df_classification['domain'].values
)
class_weight_dict = {cls: round(w, 4) for cls, w in zip(domain_classes, class_weights_array)}

print(f"\n  Domain class_weight (balanced):")
for cls, w in sorted(class_weight_dict.items(), key=lambda x: x[1], reverse=True):
    print(f"    {cls}: {w}")

# 저장 (학습 시 활용)
with open(OUTPUT_PATH / 'class_weights.json', 'w', encoding='utf-8') as f:
    json.dump(class_weight_dict, f, ensure_ascii=False, indent=2)
print(f"\n  class_weights.json 저장 완료")

=== Intent 정제 ===


  원본 intent 종류: 35,136개
  50건 이상 intent: 1080개


  정제 후 intent 종류: 1081개

상위 intent 분포:
intent_clean
기타         556029
상담사의도파악      4046
코로나 문의       2783
방송상품주문       2699
권한승인요청       2291
반품요청         1771
코로나정보        1424
자동차보험        1398
의료기관확인       1359
가격 문의        1318
상품주문신청       1296
방역지침         1267
방송상품문의       1214
시스템오류문의      1202
지원금          1198
Name: count, dtype: int64

=== Domain 클래스 불균형 분석 ===
  최대: K쇼핑 (371,210건)
  최소: 부동산 (286건)
  불균형 비율: 1298:1



  Domain class_weight (balanced):
    부동산: 185.9071
    관광여가오락: 23.7258
    학원: 19.4546
    숙박: 15.5603
    카페: 13.5671
    부동산업: 11.8312
    생활서비스: 8.2369
    소매: 6.9813
    음식점: 6.9412
    의복의류점: 6.8376
    다산콜센터: 1.0242
    금융/보험: 0.523
    질병관리본부: 0.3073
    K쇼핑: 0.1432

  class_weights.json 저장 완료


---
## 4. 레이블 인코딩

In [8]:
from sklearn.preprocessing import LabelEncoder

# 레이블 인코더 생성 및 적용
# intent → intent_clean 사용 (35K → 정제된 클래스)
label_encoders = {}

encode_cols = {
    'domain': 'domain',
    'category': 'category',
    'intent': 'intent_clean',  # 정제된 intent 사용
}

for label_name, col_name in encode_cols.items():
    le = LabelEncoder()
    df_classification[f'{label_name}_id'] = le.fit_transform(
        df_classification[col_name].fillna('unknown')
    )
    label_encoders[label_name] = le
    print(f"{label_name} (from '{col_name}'): {len(le.classes_)} classes")

# 레이블 매핑 저장
label_mapping = {
    col: {int(i): label for i, label in enumerate(le.classes_)}
    for col, le in label_encoders.items()
}

with open(OUTPUT_PATH / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print(f"\n레이블 매핑 저장: {OUTPUT_PATH / 'label_mapping.json'}")

domain (from 'domain'): 14 classes


category (from 'category'): 63 classes
intent (from 'intent_clean'): 1081 classes

레이블 매핑 저장: /Users/kuka/CIVILCOMPLAINT/data/processed/label_mapping.json


---
## 5. Train / Val / Test 분할

In [9]:
# Stratified Split (domain 기준)
# Train: 80%, Val: 10%, Test: 10%

# 먼저 Train과 나머지 분리
train_df, temp_df = train_test_split(
    df_classification,
    test_size=0.2,
    stratify=df_classification['domain_id'],
    random_state=42
)

# Val과 Test 분리
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['domain_id'],
    random_state=42
)

print(f"Train: {len(train_df):,} ({len(train_df)/len(df_classification)*100:.1f}%)")
print(f"Val:   {len(val_df):,} ({len(val_df)/len(df_classification)*100:.1f}%)")
print(f"Test:  {len(test_df):,} ({len(test_df)/len(df_classification)*100:.1f}%)")

Train: 595,497 (80.0%)
Val:   74,437 (10.0%)
Test:  74,438 (10.0%)


In [10]:
# 분류 데이터 저장 - Parquet
train_df.to_parquet(OUTPUT_PATH / 'train_classification.parquet', index=False)
val_df.to_parquet(OUTPUT_PATH / 'val_classification.parquet', index=False)
test_df.to_parquet(OUTPUT_PATH / 'test_classification.parquet', index=False)

# 레이블 인코더 저장 - Joblib
import joblib
joblib.dump(label_encoders, OUTPUT_PATH / 'label_encoders.joblib')

print(f"분류 데이터 저장 완료:")
print(f"  - train_classification.parquet ({len(train_df):,}건)")
print(f"  - val_classification.parquet ({len(val_df):,}건)")
print(f"  - test_classification.parquet ({len(test_df):,}건)")
print(f"  - label_encoders.joblib")

분류 데이터 저장 완료:
  - train_classification.parquet (595,497건)
  - val_classification.parquet (74,437건)
  - test_classification.parquet (74,438건)
  - label_encoders.joblib


---
## 6. RAG용 QA 데이터셋 생성

In [11]:
def create_qa_pairs(df):
    """콜센터 데이터에서 QA 쌍 추출 (화자 기반 텍스트 추출 개선)"""
    qa_pairs = []
    
    dialogue_id_col = '대화셋일련번호'
    turn_col = '문장번호'
    qa_col = 'QA'
    domain_col = '도메인'
    category_col = '카테고리'
    
    required_cols = [dialogue_id_col, turn_col, qa_col]
    if not all(col in df.columns for col in required_cols):
        print(f"필수 컬럼 없음. 사용 가능: {df.columns.tolist()}")
        return pd.DataFrame()
    
    def extract_q_text(row):
        """Q 레코드에서 질문 텍스트 추출 (화자 기반)"""
        if row['화자'] == '고객':
            text = row.get('고객질문(요청)', '')
            if not text or not str(text).strip():
                text = row.get('고객답변', '')
        else:
            text = row.get('상담사질문(요청)', '')
            if not text or not str(text).strip():
                text = row.get('상담사답변', '')
        return preprocessor.clean_text(text)
    
    def extract_a_text(row):
        """A 레코드에서 답변 텍스트 추출 (화자 기반)"""
        if row['화자'] == '상담사':
            text = row.get('상담사답변', '')
            if not text or not str(text).strip():
                text = row.get('상담사질문(요청)', '')
        else:
            text = row.get('고객답변', '')
            if not text or not str(text).strip():
                text = row.get('고객질문(요청)', '')
        return preprocessor.clean_text(text)
    
    # 대화별로 처리
    for dialogue_id, group in tqdm(df.groupby(dialogue_id_col), desc='Creating QA pairs'):
        group = group.sort_values(turn_col).reset_index(drop=True)
        
        for i in range(len(group) - 1):
            current = group.iloc[i]
            next_row = group.iloc[i + 1]
            
            # Q → A 쌍 찾기
            if current[qa_col] == 'Q' and next_row[qa_col] == 'A':
                q_text = extract_q_text(current)
                a_text = extract_a_text(next_row)
                
                if len(q_text) >= 5 and len(a_text) >= 5:
                    qa_pairs.append({
                        'question': q_text,
                        'answer': a_text,
                        'domain': current.get(domain_col, 'unknown'),
                        'category': current.get(category_col, 'unknown'),
                        'dialogue_id': dialogue_id
                    })
    
    df_qa = pd.DataFrame(qa_pairs)
    
    if len(df_qa) > 0:
        before = len(df_qa)
        df_qa = df_qa.drop_duplicates(subset=['question', 'answer'])
        print(f"중복 제거: {before:,} → {len(df_qa):,}")
    
    return df_qa

# QA 쌍 생성
df_qa = create_qa_pairs(df_callcenter)
print(f"\n생성된 QA 쌍: {len(df_qa):,}")


Creating QA pairs:   0%|          | 0/78998 [00:00<?, ?it/s]


Creating QA pairs:   0%|          | 1/78998 [00:00<13:34:07,  1.62it/s]


Creating QA pairs:   0%|          | 55/78998 [00:00<12:49, 102.58it/s] 


Creating QA pairs:   0%|          | 109/78998 [00:00<06:45, 194.62it/s]


Creating QA pairs:   0%|          | 158/78998 [00:00<05:00, 262.55it/s]

...


Creating QA pairs: 100%|█████████▉| 78877/78998 [02:39<00:00, 444.28it/s]


Creating QA pairs: 100%|█████████▉| 78922/78998 [02:39<00:00, 435.77it/s]


Creating QA pairs: 100%|█████████▉| 78970/78998 [02:39<00:00, 447.60it/s]


Creating QA pairs: 100%|██████████| 78998/78998 [02:39<00:00, 494.09it/s]

중복 제거: 561,048 → 456,017

생성된 QA 쌍: 456,017


In [12]:
# QA 데이터 저장 (RAG용)
if len(df_qa) > 0:
    df_qa.to_parquet(OUTPUT_PATH / 'qa_pairs.parquet', index=False)
    
    qa_documents = []
    for _, row in df_qa.iterrows():
        qa_documents.append({
            'content': f"질문: {row['question']}\n답변: {row['answer']}",
            'metadata': {
                'domain': row['domain'],
                'category': row['category'],
                'type': 'qa_pair'
            }
        })
    
    with open(OUTPUT_PATH / 'qa_documents.json', 'w', encoding='utf-8') as f:
        json.dump(qa_documents, f, ensure_ascii=False, indent=2)
    
    print(f"QA 데이터 저장 완료:")
    print(f"  - qa_pairs.parquet ({len(df_qa):,}건)")
    print(f"  - qa_documents.json")

QA 데이터 저장 완료:
  - qa_pairs.parquet (456,017건)
  - qa_documents.json


---
## 7. LLM 파인튜닝용 데이터셋 생성

In [13]:
def load_llm_instructions(path, max_files=None):
    """LLM Instruction 데이터 로드 및 변환"""
    instructions = []
    json_files = list(path.rglob('*.json'))[:max_files]
    
    for file in tqdm(json_files, desc='Loading LLM instructions'):
        try:
            with open(file, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)
                
                if isinstance(raw_data, list):
                    data = raw_data[0] if raw_data else {}
                else:
                    data = raw_data
                
                if 'instructions' in data:
                    for inst_group in data['instructions']:
                        tuning_type = inst_group.get('tuning_type', 'unknown')
                        
                        if 'data' in inst_group:
                            for item in inst_group['data']:
                                instruction = item.get('instruction', '')
                                input_text = item.get('input', '')
                                output_text = item.get('output', '')
                                
                                if instruction and output_text:
                                    instructions.append({
                                        'instruction': instruction,
                                        'input': input_text,
                                        'output': output_text,
                                        'task_type': tuning_type,
                                        'category': data.get('consulting_category', 'unknown')
                                    })
        except Exception as e:
            pass
    
    return pd.DataFrame(instructions)

# LLM 데이터 로드
df_llm = load_llm_instructions(LLM_PATH, max_files=None)
print(f"\n로드된 LLM instruction: {len(df_llm):,}")


Loading LLM instructions:   0%|          | 0/119182 [00:00<?, ?it/s]


Loading LLM instructions:   0%|          | 182/119182 [00:00<01:05, 1818.66it/s]


Loading LLM instructions:   0%|          | 373/119182 [00:00<01:03, 1871.67it/s]


Loading LLM instructions:   0%|          | 561/119182 [00:00<01:14, 1600.09it/s]


Loading LLM instructions:   1%|          | 772/119182 [00:00<01:06, 1780.06it/s]

...


Loading LLM instructions: 100%|█████████▉| 118604/119182 [00:51<00:00, 2338.29it/s]


Loading LLM instructions: 100%|█████████▉| 118839/119182 [00:51<00:00, 2319.53it/s]


Loading LLM instructions: 100%|█████████▉| 119072/119182 [00:51<00:00, 2293.93it/s]


Loading LLM instructions: 100%|██████████| 119182/119182 [00:51<00:00, 2303.24it/s]


로드된 LLM instruction: 119,182


In [14]:
# LLM 데이터: 필터링 → 검증 → 저장 (단일 파이프라인)
if len(df_llm) > 0:
    print(f"원본 LLM instruction: {len(df_llm):,}건")
    
    # 1) 유효성 필터링 (instruction, output 각각 5자 이상)
    df_llm = df_llm[
        (df_llm['instruction'].str.len() > 5) & 
        (df_llm['output'].str.len() > 5)
    ].copy()
    print(f"필터링 후: {len(df_llm):,}건")
    
    # Task Type 분포 확인
    print(f"\nTask Type 분포:")
    print(df_llm['task_type'].value_counts())
    
    # 2) Alpaca 포맷 변환
    def to_alpaca_format(row):
        if row['input']:
            return f"""### Instruction:\n{row['instruction']}\n\n### Input:\n{row['input']}\n\n### Response:\n{row['output']}"""
        else:
            return f"""### Instruction:\n{row['instruction']}\n\n### Response:\n{row['output']}"""
    
    df_llm['alpaca_text'] = df_llm.apply(to_alpaca_format, axis=1)
    
    print(f"\n=== Alpaca 포맷 샘플 ===")
    print(df_llm['alpaca_text'].iloc[0][:300])
    
    # 3) Train/Val 분할 (90/10) - 1회만 수행
    train_llm, val_llm = train_test_split(df_llm, test_size=0.1, random_state=42)
    
    # 4) 저장 (Parquet + JSONL)
    train_llm.to_parquet(OUTPUT_PATH / 'train_llm.parquet', index=False)
    val_llm.to_parquet(OUTPUT_PATH / 'val_llm.parquet', index=False)
    
    with open(OUTPUT_PATH / 'train_llm.jsonl', 'w', encoding='utf-8') as f:
        for _, row in train_llm.iterrows():
            f.write(json.dumps({
                'instruction': row['instruction'],
                'input': row['input'],
                'output': row['output']
            }, ensure_ascii=False) + '\n')
    
    print(f"\nLLM 데이터 저장 완료:")
    print(f"  - train_llm.parquet ({len(train_llm):,}건)")
    print(f"  - val_llm.parquet ({len(val_llm):,}건)")
    print(f"  - train_llm.jsonl ({len(train_llm):,}건)")

원본 LLM instruction: 119,182건
필터링 후: 85,417건

Task Type 분포:
task_type
요약      35848
분류      31779
질의응답    17790
Name: count, dtype: int64



=== Alpaca 포맷 샘플 ===
### Instruction:
다음 상담의 주제는 "상품 및 서비스 일반", "주문/결제/입금 확인", "취소/반품/교환/환불/AS", "배송 문의", "회원 관리", "제휴", "이벤트/할인", "콘텐츠", "기타" 중 무엇일까요?

### Input:
고객: 안녕하세요. ▲/▲▲부터 ▲▲일까지 4박 오션뷰로 예약하고 싶은데, 코로나 때문에 여행이 금지되면 환불은 어떻게 되나요?
상담사: 안녕하세요! ▲/▲▲부터 ▲▲일까지 더블식스 디럭스스윗 오션뷰 4박(2+2) 가격은 $920입니다. 발리에서는 코로나로 인한 특별 규정이 없어서



LLM 데이터 저장 완료:
  - train_llm.parquet (76,875건)
  - val_llm.parquet (8,542건)
  - train_llm.jsonl (76,875건)


In [15]:
# 축소 원인 분석
q_only = df_callcenter[df_callcenter['QA'] == 'Q']
print(f"Q만 필터링: {len(q_only):,}")

# 텍스트 길이 분포
q_only['text_len'] = q_only['고객질문(요청)'].fillna('').str.len()
print(f"길이 0인 것: {(q_only['text_len'] == 0).sum():,}")
print(f"길이 1~4인 것: {((q_only['text_len'] > 0) & (q_only['text_len'] < 5)).sum():,}")
print(f"길이 5 이상: {(q_only['text_len'] >= 5).sum():,}")

Q만 필터링: 1,014,311

길이 0인 것: 623,191
길이 1~4인 것: 2,360
길이 5 이상: 388,760


In [16]:
# 길이 0인 Q 레코드 분석
q_empty = df_callcenter[(df_callcenter['QA'] == 'Q') & 
                         (df_callcenter['고객질문(요청)'].fillna('').str.len() == 0)]

print(f"=== 길이 0인 Q 레코드 분석 ({len(q_empty):,}건) ===\n")

# 화자 분포 확인
print("화자 분포:")
print(q_empty['화자'].value_counts())

# 다른 텍스트 컬럼에 데이터가 있는지 확인
print("\n각 컬럼별 비어있지 않은 비율:")
text_cols = ['고객질문(요청)', '고객답변', '상담사질문(요청)', '상담사답변']
for col in text_cols:
    if col in q_empty.columns:
        non_empty = (q_empty[col].fillna('').str.len() > 0).sum()
        print(f"  {col}: {non_empty:,}건 ({non_empty/len(q_empty)*100:.1f}%)")

# 샘플 5개 확인
print("\n샘플 레코드:")
sample_cols = ['화자', 'QA', '고객의도', '고객질문(요청)', '상담사질문(요청)']
print(q_empty[sample_cols].head(5).to_string())

=== 길이 0인 Q 레코드 분석 (623,191건) ===

화자 분포:
화자
상담사    623180
고객         11
Name: count, dtype: int64

각 컬럼별 비어있지 않은 비율:
  고객질문(요청): 0건 (0.0%)


  고객답변: 194건 (0.0%)


  상담사질문(요청): 623,181건 (100.0%)
  상담사답변: 755건 (0.1%)

샘플 레코드:


     화자 QA 고객의도 고객질문(요청)                                                          상담사질문(요청)
2   상담사  Q                               아 그러세요. 정보 확인 후에 도와 드리겠습니다. 성함하고 전화번호 말씀 부탁 드립니다. 
4   상담사  Q                                                     소중한 정보 확인 감사합니다. 어느 상품이십니까? 
6   상담사  Q                                          ㅇㅇㅇㅇ 에어쿠션 컴포트 여성화 블랙 이백사십 주문하신 거 확인됩니다.
8   상담사  Q                                                    실례지만 교환하시는 사유가 사이즈가 작으신 건가요? 
10  상담사  Q                그러면 고객님 블랙 이백사십오로 교환해 드리겠습니다. 배송 받으신 상품 택 제거하시거나 외부 착화는 안 하셨습니까? 


---
## 8. 전처리 결과 요약

In [17]:
# 최종 요약 (개선 반영)
summary = {
    'classification': {
        'train': len(train_df),
        'val': len(val_df),
        'test': len(test_df),
        'total': len(df_classification),
        'num_domains': df_classification['domain'].nunique(),
        'num_categories': df_classification['category'].nunique(),
        'num_intents_raw': df_classification['intent'].nunique(),
        'num_intents_clean': df_classification['intent_clean'].nunique(),
        'intent_min_count': INTENT_MIN_COUNT,
        'class_weight_applied': True,
        'text_len_range': '5~500'
    },
    'qa_pairs': len(df_qa) if 'df_qa' in dir() else 0,
    'llm_instructions': {
        'train': len(train_llm) if 'train_llm' in dir() else 0,
        'val': len(val_llm) if 'val_llm' in dir() else 0,
        'filtered_from': 'instruction/output 각 5자 이상'
    }
}

print("=" * 60)
print("전처리 완료 요약 (개선)")
print("=" * 60)
print(f"\n[분류 데이터]")
print(f"  Train: {summary['classification']['train']:,}")
print(f"  Val:   {summary['classification']['val']:,}")
print(f"  Test:  {summary['classification']['test']:,}")
print(f"  도메인 수: {summary['classification']['num_domains']}")
print(f"  카테고리 수: {summary['classification']['num_categories']}")
print(f"  Intent (원본): {summary['classification']['num_intents_raw']:,}개")
print(f"  Intent (정제): {summary['classification']['num_intents_clean']}개 (>={INTENT_MIN_COUNT}건)")
print(f"  텍스트 길이: {summary['classification']['text_len_range']}")
print(f"  class_weight: class_weights.json 저장됨")
print(f"\n[RAG QA 쌍]: {summary['qa_pairs']:,}")
print(f"\n[LLM 파인튜닝 데이터]")
print(f"  Train: {summary['llm_instructions']['train']:,}")
print(f"  Val:   {summary['llm_instructions']['val']:,}")

# 요약 저장
with open(OUTPUT_PATH / 'preprocessing_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print(f"\n저장 위치: {OUTPUT_PATH}")
print(f"\n생성된 파일:")
for f in sorted(OUTPUT_PATH.glob('*')):
    if f.name != '.gitkeep':
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.1f}MB)")

전처리 완료 요약 (개선)

[분류 데이터]
  Train: 595,497
  Val:   74,437
  Test:  74,438
  도메인 수: 14
  카테고리 수: 63
  Intent (원본): 35,136개
  Intent (정제): 1081개 (>=50건)
  텍스트 길이: 5~500
  class_weight: class_weights.json 저장됨

[RAG QA 쌍]: 456,017

[LLM 파인튜닝 데이터]
  Train: 76,875
  Val:   8,542

저장 위치: /Users/kuka/CIVILCOMPLAINT/data/processed

생성된 파일:
  class_weights.json (0.0MB)
  exploration_summary.json (0.0MB)
  label_encoders.joblib (0.0MB)
  label_mapping.json (0.0MB)
  preprocessing_summary.json (0.0MB)
  qa_documents.json (119.7MB)
  qa_pairs.parquet (24.8MB)
  test_classification.parquet (3.6MB)
  train_classification.parquet (28.1MB)
  train_llm.csv (487.4MB)
  train_llm.jsonl (246.7MB)
  train_llm.parquet (205.2MB)
  val_classification.parquet (3.6MB)
  val_llm.csv (54.3MB)
  val_llm.parquet (22.8MB)


---
## 9. Base64 다운로드 (Kaggle)

In [ ]:
import base64, io, zipfile
from IPython.display import display, HTML

def create_download_link(filepath, filename=None):
    """파일을 Base64로 인코딩하여 다운로드 링크 생성 (Kaggle용)"""
    filepath = str(filepath)
    if filename is None:
        filename = os.path.basename(filepath)
    if not os.path.exists(filepath):
        print(f"⚠️ 파일 없음: {filepath}")
        return
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / (1024 * 1024)
    href = f'<a href="data:application/octet-stream;base64,{b64}" download="{filename}">📥 {filename} ({size_mb:.1f} MB)</a>'
    display(HTML(href))

def create_zip_download(file_dict, zip_name="artifacts.zip"):
    """여러 파일을 ZIP으로 묶어 Base64 다운로드"""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for arcname, filepath in file_dict.items():
            filepath = str(filepath)
            if os.path.exists(filepath):
                zf.write(filepath, arcname)
            else:
                print(f"⚠️ 스킵: {filepath}")
    buffer.seek(0)
    data = buffer.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / (1024 * 1024)
    href = f'<a href="data:application/octet-stream;base64,{b64}" download="{zip_name}">📦 {zip_name} ({size_mb:.1f} MB)</a>'
    display(HTML(href))

# 전처리 결과물 다운로드
print("=== 전처리 아티팩트 다운로드 ===\n")

# 개별 파일
for fname in [
    'label_mapping.json', 'class_weights.json', 'label_encoders.joblib',
    'preprocessing_summary.json',
    'train_classification.parquet', 'val_classification.parquet', 'test_classification.parquet',
    'qa_pairs.parquet', 'qa_documents.json',
    'train_llm.parquet', 'val_llm.parquet', 'train_llm.jsonl',
]:
    fpath = OUTPUT_PATH / fname
    if fpath.exists():
        create_download_link(fpath)
    else:
        print(f"⚠️ 없음: {fname}")

print("\n✅ 다운로드 링크 생성 완료")